# BERTurk Multi-Label Intent Training

This notebook trains `dbmdz/bert-base-turkish-cased` on `deprem-private/intent_test_v13_anonymized`
using the dataset's original multi-label intent annotations.


In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy


In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


In [ ]:
MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
DATASET_NAME = 'deprem-private/intent_test_v13_anonymized'
MAX_LENGTH = 256
TEST_SIZE = 0.15
RANDOM_SEED = 42
THRESHOLD = 0.5


In [ ]:
dataset = load_dataset(DATASET_NAME)
dataset


In [ ]:
all_labels = set()
for row in dataset['train']:
    for label in row['label']:
        all_labels.add(label)

label_list = sorted(all_labels)
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

print('Label sayısı:', len(label_list))
print(label_list)


In [ ]:
def encode_example(example):
    multi_hot = [0.0] * len(label_list)
    for label in example['label']:
        multi_hot[label2id[label]] = 1.0
    example['labels'] = multi_hot
    example['text'] = example['image_url']
    return example

dataset = dataset.map(encode_example)
dataset['train'][0]


In [ ]:
split_dataset = dataset['train'].train_test_split(test_size=TEST_SIZE, seed=RANDOM_SEED)
train_ds = split_dataset['train']
val_ds = split_dataset['test']

print(train_ds)
print(val_ds)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

keep_cols = ['input_ids', 'attention_mask', 'labels']
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])

train_ds.set_format('torch')
val_ds.set_format('torch')


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    problem_type='multi_label_classification',
)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)

    return {
        'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'samples_f1': f1_score(labels, preds, average='samples', zero_division=0),
        'subset_accuracy': accuracy_score(labels, preds),
    }


In [ ]:
training_args = TrainingArguments(
    output_dir='./berturk-deprem-intent',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='micro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()


In [ ]:
metrics = trainer.evaluate()
metrics


In [ ]:
trainer.save_model('./berturk-deprem-intent-final')
tokenizer.save_pretrained('./berturk-deprem-intent-final')


In [ ]:
text = 'Acil çadır ve battaniye ihtiyacı var. Hatay Antakya Odabaşı Mahallesi'
inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LENGTH)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.sigmoid(logits).cpu().numpy()[0]

predicted_labels = [label_list[i] for i, p in enumerate(probs) if p >= THRESHOLD]
print('Tahmin edilen label'lar:', predicted_labels)
print('Skorlar:')
for i, p in enumerate(probs):
    if p >= 0.2:
        print(label_list[i], round(float(p), 4))
